# 15 · Completed AIA 72-hour training

**Fitting completed on 4 October 2026 at 14:37 UTC.** All three fixed seeds finished under the original stopping rule. Seed 17 completed six epochs and selected epoch 2; seeds 29 and 43 each completed seven epochs and selected epoch 3.

This is a saved executed notebook. Its six code cells and outputs are preserved from the VM execution. Only this introductory Markdown and provenance metadata have been added after archiving. The archive retains the unmodified executed notebook.

## What these results establish
The CPU audit verified unchanged inputs and the first seed, checkpoint integrity, finite saved tensors, the exact first early-stopping boundary for seeds 29 and 43, and all 14 saved selection losses. Saved-best GPU replay for these two seeds is the next step. Seed 17's accepted replay is retained.

## Research limits
This is completed fitting on the candidate 72-hour target, not completed later-period evaluation, calibration, uncertainty analysis or SHARP/AIA fusion. Training uses 2010–2013; model selection uses January–June 2014. Labels and historical data availability remain provisional. Chronological separation alone does not establish AR/event-disjoint validation.

## Configuration and original execution
The cells below preserve the frozen configuration, original CNN–GRU training method and recorded results.

In [1]:
from pathlib import Path
import json, os, sys, time, math, random
import numpy as np
import pandas as pd
import torch
from torch import nn

if 'CONTINUATION_CONTEXT' not in globals():
    candidates = [Path.cwd(), *Path.cwd().parents]
    candidates += [p/'outputs/aia72_completion_training_v5_20261004/bundle' for p in candidates]
    ROOT = next(p for p in candidates if (p/'configs/aia72_continuation_v1.json').is_file()
                and json.loads((p/'configs/aia72_continuation_v1.json').read_text()).get('initial_seed29_sha256')
                == '455e7be13bec0d831f827f8a4b2f7aa64c19781bab480c8d3ef076466ba7dc95')
    CONTINUATION_CONTEXT = {'mode':'preview','bundle_root':str(ROOT),
                            'preview_source_root':str(ROOT.parent/'source_mirror'),
                            'preview_initial':str(ROOT.parent/'initial_training')}
ROOT=Path(CONTINUATION_CONTEXT['bundle_root'])
sys.path.insert(0,str(ROOT))
import scripts.aia72_continue as continuation
import scripts.train_aia72_gpu as original
from scripts.aia72_continue_contract import check_contract
from scripts.aia72_replay_contract import require
from scripts.train_aia72_gpu import atomic_save,rng_state,restore_rng,set_seed,loader,validation_predictions
from scripts.aia72_vm_data import write_json
from scripts.dataset_io import file_sha256
CONTRACT=check_contract(json.loads((ROOT/'configs/aia72_continuation_v1.json').read_text()))
display(pd.DataFrame([{'Mode':CONTINUATION_CONTEXT['mode'],'First seed':29,'Following seed':43,
 'Maximum slot hours':36,'Maximum fitting hours':35,'Seed17 repeated':False}]))

Mode,First seed,Following seed,Maximum slot hours,Maximum fitting hours,Seed17 repeated
authorized_training_continuation,29,43,36,35,False


## Data
### 2. Verify the saved starting point
Verify metadata and checkpoint hashes without reading any image pixels. The snapshot remains immutable; the live continuation writes to a separate copy.

In [2]:
preview=CONTINUATION_CONTEXT['mode']=='preview'
source=CONTINUATION_CONTEXT.get('preview_source_root',CONTRACT['source_root'])
initial=CONTINUATION_CONTEXT.get('preview_initial',str(Path(CONTRACT['review_root'])/'initial_training'))
frame,records,normalization,CONFIG,state=continuation.inspect_inputs(CONTRACT,source,initial)
display(pd.DataFrame([{'Seed':29,'Next epoch':state['epoch'],'Processed cases in epoch':state['cursor'],
                      'Optimizer steps':state['steps'],'Completed epochs':len(state['history'])}]))
display(frame.groupby('role',sort=False).agg(Cases=('forecast_case_id','size'),Positive_windows=('label','sum')).reset_index())
print('Saved normalization reused. Image pixels read in this cell: 0.')

Seed,Next epoch,Processed cases in epoch,Optimizer steps,Completed epochs
29,7,20416,10876,6


role,Cases,Positive_windows
train,25586,1476
model_validation,3905,416


Saved normalization reused. Image pixels read in this cell: 0.


### 3. Original model architecture
This executable definition is unchanged from the frozen training source.

In [3]:
"""AIA CNN-GRU architecture reused from Watchman77/solar-flare-aia-training.

The three classes are preserved from 19A2_Temporal_AIA_CNN_GRU_Cycle24_Training.ipynb.
Only the module wrapper is new. No upstream weights or normalization are reused.
The new 72-hour label/role contract is supplied by this repository's loader.
Provenance: results/aia72_canary_20261002/source_provenance.json.
"""

import torch
from torch import nn

DROPOUT = 0.30

class ConvBlock(nn.Module):
    def __init__(self, cin, cout, dropout=0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(cin, cout, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(cout),
            nn.GELU(),
            nn.Conv2d(cout, cout, 3, padding=1, bias=False),
            nn.BatchNorm2d(cout),
            nn.GELU(),
            nn.Dropout2d(dropout) if dropout > 0 else nn.Identity(),
        )

    def forward(self, x):
        return self.net(x)

class FrameCNN(nn.Module):
    def __init__(self, embed_dim=256):
        super().__init__()
        self.encoder = nn.Sequential(
            ConvBlock(6, 32, 0.05),
            ConvBlock(32, 64, 0.05),
            ConvBlock(64, 128, 0.10),
            ConvBlock(128, 192, 0.10),
            nn.AdaptiveAvgPool2d(1),
        )
        self.proj = nn.Sequential(
            nn.Flatten(),
            nn.Linear(192, embed_dim),
            nn.GELU(),
            nn.Dropout(DROPOUT),
        )

    def forward(self, x):
        return self.proj(self.encoder(x))

class TemporalAIACNNGRU(nn.Module):
    def __init__(self, embed_dim=256, hidden_dim=192):
        super().__init__()
        self.frame_encoder = FrameCNN(embed_dim)
        self.gru = nn.GRU(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True,
        )
        self.head = nn.Sequential(
            nn.LayerNorm(hidden_dim),
            nn.Dropout(DROPOUT),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, x):
        b,t,c,h,w = x.shape
        z = self.frame_encoder(x.reshape(b*t,c,h,w)).reshape(b,t,-1)
        _, hlast = self.gru(z)
        return self.head(hlast[-1]).squeeze(-1)


### 4. Original resumable training loop
The definition below is the unchanged original function. The worker calls that same hash-verified function from its frozen module. Its only loader wrapper passes the shared lock into spawned workers; sample order, generators, preprocessing and optimizer settings are unchanged. The original six-hour scheduler is not called; the new outer controller enforces the separately recorded continuous fitting allowance.

In [4]:
def train_seed(train, validation, config, seed, output, contract_sha256, device, deadline,
               model_factory=TemporalAIACNNGRU, stop_after_steps=None):
    """Save next-case cursor and all RNG/optimizer state for an exact restart."""
    output = Path(output); output.mkdir(parents=True, exist_ok=True)
    checkpoint = output / f'seed_{seed}_resume.pt'
    completion = output / f'seed_{seed}_complete.json'
    if completion.exists():
        done = json.loads(completion.read_text())
        if done['contract_sha256'] != contract_sha256 or file_sha256(output / f'seed_{seed}_best.pt') != done['best_checkpoint_sha256']:
            raise ValueError('Completed seed contract/checkpoint changed')
        return done
    set_seed(seed)
    model = model_factory().to(device)
    prior = float(train.frame.label.mean())
    if not 0 < prior < 1:
        raise ValueError('Training needs both outcome classes')
    with torch.no_grad():
        model.head[-1].bias.fill_(math.log(prior / (1 - prior)))
    optimizer = torch.optim.AdamW(model.parameters(), lr=config['learning_rate'], weight_decay=config['weight_decay'])
    state = {'epoch': 1, 'cursor': 0, 'loss_sum': 0., 'best_loss': None, 'best_epoch': None,
             'stale': 0, 'history': [], 'elapsed_seconds': 0., 'steps': 0}
    if checkpoint.exists():
        saved = torch.load(checkpoint, map_location=device, weights_only=True)
        if saved['contract_sha256'] != contract_sha256:
            raise ValueError('Cannot resume under changed code/data/configuration')
        state = saved['state']; model.load_state_dict(saved['model']); optimizer.load_state_dict(saved['optimizer'])
        restore_rng(saved['rng'])
    start = time.monotonic(); previous_elapsed = state['elapsed_seconds']; last_save = start
    criterion = nn.BCEWithLogitsLoss()
    def save(status='running'):
        nonlocal last_save
        state['elapsed_seconds'] = previous_elapsed + time.monotonic() - start
        atomic_save({'contract_sha256': contract_sha256, 'state': state, 'model': model.state_dict(),
                     'optimizer': optimizer.state_dict(), 'rng': rng_state()}, checkpoint)
        write_json(output / 'progress.json', {'utc': datetime.now(timezone.utc).isoformat(), 'status': status,
                   'seed': seed, 'epoch': state['epoch'], 'case_cursor': state['cursor'], 'training_cases': len(train),
                   'steps': state['steps'], 'elapsed_seconds': state['elapsed_seconds'],
                   'contract_sha256': contract_sha256, 'full_training_complete': False})
        last_save = time.monotonic()
    save()
    while state['epoch'] <= config['max_epochs'] and state['stale'] < config['patience']:
        epoch = state['epoch']
        if not 0 <= state['cursor'] <= len(train) or (state['cursor'] % config['batch_size'] and state['cursor'] != len(train)):
            raise ValueError('Invalid restart cursor')
        model.train()
        for images, labels, identities in loader(train, config, seed, epoch, state['cursor'], shuffled=True):
            if time.monotonic() >= deadline or (output / 'STOP_REQUESTED').exists():
                save('checkpointed_incomplete'); return {'status': 'checkpointed_incomplete', 'seed': seed}
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(images.to(device, non_blocking=True)), labels.to(device, non_blocking=True))
            if not torch.isfinite(loss):
                raise ValueError('Nonfinite training loss')
            loss.backward()
            norm = nn.utils.clip_grad_norm_(model.parameters(), config['gradient_clip'], error_if_nonfinite=True)
            optimizer.step()
            state['cursor'] += len(labels); state['loss_sum'] += float(loss.detach()) * len(labels); state['steps'] += 1
            if time.monotonic() - last_save >= config['checkpoint_interval_seconds']:
                save(); print(json.dumps({'seed': seed, 'epoch': epoch, 'cases': state['cursor'], 'of': len(train)}), flush=True)
            if stop_after_steps is not None and state['steps'] >= stop_after_steps:
                save('test_interruption'); return {'status': 'test_interruption', 'seed': seed}
        save()
        predictions = validation_predictions(model, validation, config, seed, device, deadline)
        if predictions is None:
            save('checkpointed_incomplete'); return {'status': 'checkpointed_incomplete', 'seed': seed}
        y, logits = predictions.label.to_numpy(), predictions.logit.to_numpy()
        val_loss = float(np.mean(np.logaddexp(0., logits) - y * logits))
        if not math.isfinite(val_loss):
            raise ValueError('Invalid model-selection loss')
        predictions['probability'] = 1 / (1 + np.exp(-np.clip(logits, -700, 700)))
        predictions.to_csv(output / f'seed_{seed}_epoch_{epoch:02d}_selection.csv.gz', index=False,
                           compression={'method': 'gzip', 'mtime': 0})
        if state['best_loss'] is None or val_loss < state['best_loss']:
            state['best_loss'], state['best_epoch'], state['stale'] = val_loss, epoch, 0
            atomic_save(model.state_dict(), output / f'seed_{seed}_best.pt')
        else:
            state['stale'] += 1
        row = {'seed': seed, 'epoch': epoch, 'training_log_loss': state['loss_sum'] / len(train),
               'selection_log_loss': val_loss, 'best_epoch': state['best_epoch']}
        state['history'].append(row); print(json.dumps(row), flush=True)
        pd.DataFrame(state['history']).to_csv(output / f'seed_{seed}_history.csv', index=False)
        state['epoch'] += 1; state['cursor'] = 0; state['loss_sum'] = 0.; save()
    done = {'status': 'seed_fit_complete_pending_replay', 'seed': seed, 'contract_sha256': contract_sha256,
            'best_epoch': state['best_epoch'], 'selection_log_loss': state['best_loss'],
            'epochs_completed': len(state['history']), 'elapsed_seconds': state['elapsed_seconds'],
            'best_checkpoint_sha256': file_sha256(output / f'seed_{seed}_best.pt')}
    write_json(completion, done)
    return done


## Results
### 5. Resume only through the reviewed controller
The supervised process group contains the notebook, model and loaders. Stop, failure, disk, ownership and deadline checks apply throughout. No automatic retry or second invocation is permitted.

In [5]:
if CONTINUATION_CONTEXT['mode']=='authorized_training_continuation':
    result=continuation.run_training(CONTRACT,CONTINUATION_CONTEXT)
else:
    require(CONTINUATION_CONTEXT['mode']=='preview','Unknown mode')
    result={'status':'prepared_only_no_training','additional_steps':0,'scientific_acceptance':False}
display(pd.DataFrame([{'Item':key,'Value':result[key]} for key in ['status','additional_steps','scientific_acceptance']]))
if result.get('checkpoints'):
    display(pd.DataFrame(result['checkpoints'])[['seed','epoch','cursor','steps','fit_complete']])

Item,Value
status,remaining_seeds_fit_pending_replay
additional_steps,11524
scientific_acceptance,False


seed,epoch,cursor,steps,fit_complete
29,8,0,11200,True
43,8,0,11200,True


{"seed": 29, "epoch": 7, "cases": 20816, "of": 25586}
{"seed": 29, "epoch": 7, "cases": 21456, "of": 25586}
{"seed": 29, "epoch": 7, "cases": 22096, "of": 25586}
{"seed": 29, "epoch": 7, "cases": 22736, "of": 25586}
{"seed": 29, "epoch": 7, "cases": 23376, "of": 25586}
{"seed": 29, "epoch": 7, "cases": 24016, "of": 25586}
{"seed": 29, "epoch": 7, "cases": 24656, "of": 25586}
{"seed": 29, "epoch": 7, "cases": 25296, "of": 25586}
{"seed": 29, "epoch": 7, "training_log_loss": 0.1682859625066593, "selection_log_loss": 0.4478900322316879, "best_epoch": 3}
{"seed": 43, "epoch": 1, "cases": 400, "of": 25586}
{"seed": 43, "epoch": 1, "cases": 1040, "of": 25586}
{"seed": 43, "epoch": 1, "cases": 1680, "of": 25586}
{"seed": 43, "epoch": 1, "cases": 2320, "of": 25586}
{"seed": 43, "epoch": 1, "cases": 2960, "of": 25586}
{"seed": 43, "epoch": 1, "cases": 3600, "of": 25586}
{"seed": 43, "epoch": 1, "cases": 4240, "of": 25586}
{"seed": 43, "epoch": 1, "cases": 4880, "of": 25586}
{"seed": 43, "epoch"

## Takeaways
### 6. Preserve the actual result
Successful fitting still requires saved-model replay and later analysis. Preserve the executed notebook, checkpoint, controller receipt and idle resource receipt together. Gray-Box retains scheduling priority across the technical boundary; AIA does not resume until the agreed analysis is complete and explicitly returned.

In [6]:
print('Notebook status:',result['status'])
print('Later-year evaluation, calibration and fusion remain pending.')

Notebook status: remaining_seeds_fit_pending_replay
Later-year evaluation, calibration and fusion remain pending.
